In [ ]:
conda create -n sti_stlearn python=3.10 -y
conda activate sti_stlearn
pip install stlearn scanpy anndata pandas numpy scipy matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import scanpy as sc
import stlearn as st
import matplotlib.pyplot as plt

BASE = Path.home() / "MERFISH" / "S1R1" / "STI_results"
OUT = BASE / "stLearn"
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(BASE / "S1R1_prepared.h5ad")
adata.X = adata.layers["counts"].copy()

# stLearn can perform expression-based trajectory inference
# without using the microscopy images.
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata, n_top_genes=min(2000, adata.n_vars), flavor="seurat"
)
adata = adata[:, adata.var["highly_variable"]].copy()

sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=min(30, adata.n_vars - 1))
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
sc.tl.leiden(adata, resolution=0.5, key_added="leiden")

print("Available clusters:", adata.obs["leiden"].value_counts().to_dict())

# Plot clusters in actual tissue coordinates to help choose a root.
xy = adata.obsm["spatial"]
fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(
    xy[:, 0], xy[:, 1],
    c=adata.obs["leiden"].astype("category").cat.codes,
    s=3, cmap="tab20"
)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("stLearn input clusters — select a root cluster")
fig.colorbar(scatter, ax=ax, label="Encoded cluster")
fig.tight_layout()
fig.savefig(OUT / "01_clusters.png", dpi=200)
plt.show()

# IMPORTANT: change this after examining the cluster plot.
ROOT_CLUSTER = "0"

if ROOT_CLUSTER not in adata.obs["leiden"].astype(str).values:
    raise ValueError("ROOT_CLUSTER not present in the Leiden labels.")

# stLearn's set_root selects a root cell from the specified cluster.
adata.uns["iroot"] = st.spatial.trajectory.set_root(
    adata,
    use_label="leiden",
    cluster=ROOT_CLUSTER,
    use_raw=False,
)

# eps is in the coordinate units used by the method's spatial workflow.
# Start with a conservative value and tune to your coordinate scale.
st.spatial.trajectory.pseudotime(
    adata,
    use_label="leiden",
    eps=20,
    n_neighbors=25,
    use_rep="X_pca",
    pseudotime_key="dpt_pseudotime",
)

# Save the cell-level pseudotime.
adata.obs.to_csv(OUT / "cell_results.csv")
adata.write_h5ad(OUT / "stlearn_results.h5ad")

# Plot pseudotime directly on the tissue coordinates.
pt = adata.obs["dpt_pseudotime"].to_numpy()
fig, ax = plt.subplots(figsize=(8, 7))
scat = ax.scatter(xy[:, 0], xy[:, 1], c=pt, s=3, cmap="viridis")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("stLearn pseudotime — S1R1")
fig.colorbar(scat, ax=ax, label="Pseudotime")
fig.tight_layout()
fig.savefig(OUT / "02_pseudotime.png", dpi=200)
plt.show()

print("stLearn results saved to:", OUT)
